<a href="https://colab.research.google.com/github/melanieesg/OT-Consulting-Exploracion/blob/main/Acondicionamiento_OT_Consulting_Etapa3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OT Consulting – Etapa 3: Tratamiento y acondicionamiento de datos





## Configuración
Se definen la fecha de corte, las horas laborales al mes (176) y la carga administrativa (1.20).

In [2]:
# OT Consulting – Etapa 3: Corrección y acondicionamiento de datos
# Aplica los métodos de corrección (Tabla 2) y las fórmulas (Tabla 3) del documento del equipo
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

FECHA_CORTE = pd.Timestamp.today().normalize()
HORAS_MES = 176          # horas laborales al mes (Tabla 3)
CARGA_ADMIN = 1.20       # carga administrativa (Tabla 3)

def mostrar(df, titulo):
    display(df.style.hide(axis="index").set_caption(titulo).set_table_styles([
        {"selector": "caption", "props": "font-size:16px; font-weight:bold; text-align:left; padding:8px 0;"},
        {"selector": "th", "props": "background:#073763; color:white; padding:8px 12px; text-align:left;"},
        {"selector": "td", "props": "padding:6px 12px; border-bottom:1px solid #ddd;"},
        {"selector": "tr:nth-child(even) td", "props": "background:#EFEFEF;"},
    ]).format(precision=2, na_rep="—"))

## 1. Importación de archivos

In [4]:
# 1. Importar archivos
archivos = {"Colaboradores": "Catalogo_Colaboradores", "Clientes": "Catalogo_Clientes",
            "Asignaciones": "Asignaciones_Colaboradores", "Matriz": "Matriz_Habilidades",
            "Habilidades": "Catalogo_Habilidades"}
if not all(Path(a + ".xlsx").exists() for a in archivos.values()):
    from google.colab import files
    files.upload()
originales = {n: pd.read_excel(a + ".xlsx") for n, a in archivos.items()}

Saving Asignaciones_Colaboradores.xlsx to Asignaciones_Colaboradores (1).xlsx
Saving Catalogo_Clientes.xlsx to Catalogo_Clientes.xlsx
Saving Catalogo_Colaboradores.xlsx to Catalogo_Colaboradores.xlsx
Saving Catalogo_Habilidades.xlsx to Catalogo_Habilidades.xlsx
Saving Matriz_Habilidades.xlsx to Matriz_Habilidades.xlsx


## 2. Validación general

In [5]:
# 2. Validación general: espacios, textos vacíos y duplicados
datos = {}
for n, df in originales.items():
    df = df.copy()
    df.columns = df.columns.str.strip()
    datos[n] = df.apply(lambda s: s.map(lambda x: x.strip() if isinstance(x, str) else x)).replace("", np.nan)
mostrar(pd.DataFrame([{"Tabla": n, "Registros": len(df), "Duplicados": int(df.duplicated().sum())}
                      for n, df in datos.items()]), "Validación general")

col, cli, asig, matriz, hab = (datos[n] for n in archivos)

Tabla,Registros,Duplicados
Colaboradores,107,0
Clientes,12,0
Asignaciones,31,0
Matriz,54,0
Habilidades,71,0


## 3. Estandarización de fechas

In [6]:
# 3. Estandarizar fechas a AAAA-MM-DD ("Indefinido" queda vacío)
asig["Fecha Fin"] = asig["Fecha Fin"].replace("Indefinido", np.nan)
for df, campos in [(asig, ["Fecha Inicio", "Fecha Fin"]), (cli, ["Fecha Inicio", "Fecha Fin"]),
                   (col, ["Fecha Incorporación", "Fecha de Baja"])]:
    for c in campos:
        df[c] = pd.to_datetime(df[c], errors="coerce")

## 4. Colaboradores: puestos y departamentos pendientes

In [7]:
# 4. Colaboradores: los PENDIENTE se conservan y se marcan
col["Puesto_Definido"] = np.where(col["Puesto"].notna() & col["Puesto"].ne("PENDIENTE"), 1, 0)
col["Departamento_Registrado"] = np.where(col["Departamento"].notna(), 1, 0)
col["Baja_Sin_Fecha"] = np.where(col["Estatus"].eq("Baja") & col["Fecha de Baja"].isna(), 1, 0)
sueldo = col["Sueldo Total Integrado"].where(col["Sueldo Total Integrado"] > 0)
col["Costo_Por_Hora"] = (sueldo / HORAS_MES).round(2)
col["Costo_Hora_Con_Carga"] = (col["Costo_Por_Hora"] * CARGA_ADMIN).round(2)

## 5. Asignaciones: estatus, días restantes y disponibilidad

In [8]:
# 5. Asignaciones: estatus, días restantes y proyectos simultáneos
asig["Estatus_Asignacion"] = np.select(
    [asig["Fecha Inicio"] > FECHA_CORTE, asig["Fecha Fin"] < FECHA_CORTE],
    ["Por iniciar", "Finalizada"], default="En curso")   # sin fecha de fin = vigente
asig["Dias_Restantes"] = (asig["Fecha Fin"] - FECHA_CORTE).dt.days
vigentes = asig[asig["Estatus_Asignacion"].eq("En curso")]
simultaneos = vigentes.groupby("ID Colaborador")["Proyecto"].nunique()
col["Proyectos_Simultaneos"] = col["ID"].map(simultaneos).fillna(0).astype(int)
col["Disponible"] = np.where(col["Estatus"].eq("Activo") & col["Proyectos_Simultaneos"].eq(0), "Sí", "No")

## 6. Integridad referencial y habilidades

In [9]:
# 6. Integridad referencial y habilidades
asig["ID_Valido"] = np.where(asig["ID Colaborador"].isin(col["ID"]), 1, 0)
col["Tiene_Habilidades"] = np.where(col["ID"].isin(matriz["ID Colaborador"]), 1, 0)
cols_hab = [c for c in matriz.columns if c.split(" - ")[0] in set(hab["codigo"])]
habilidades = matriz.melt(id_vars="ID Colaborador", value_vars=cols_hab,
                          var_name="Habilidad", value_name="Nivel").dropna(subset=["Nivel"])
habilidades["Nivel_Dominio"] = habilidades["Nivel"].map({"Básico": 1, "Intermedio": 2, "Avanzado": 3, "Experto": 4})

## 7. Tabla acondicionada

In [10]:
# 7. Tabla acondicionada (asignaciones como tabla de hechos)
tabla = asig.drop(columns="Nombre Colaborador").merge(
    col[["ID", "Estatus", "Departamento", "Puesto", "Puesto_Definido", "Costo_Por_Hora",
         "Costo_Hora_Con_Carga", "Proyectos_Simultaneos", "Tiene_Habilidades"]],
    left_on="ID Colaborador", right_on="ID", how="left").drop(columns="ID")

## 8. Evidencia: antes y después

In [11]:
# 8. Evidencia: antes y después
activos = col["Estatus"].eq("Activo")
oc, oa = originales["Colaboradores"], originales["Asignaciones"]
resumen = pd.DataFrame([
    ["Colaboradores sin departamento", oc["Departamento"].isna().sum(), "Se conservan; Departamento_Registrado = 0", (col["Departamento_Registrado"] == 0).sum()],
    ["Puestos PENDIENTE", oc["Puesto"].eq("PENDIENTE").sum(), "Se conservan; Puesto_Definido = 0", (col["Puesto_Definido"] == 0).sum()],
    ["Asignaciones sin fecha de fin", oa["Fecha Fin"].eq("Indefinido").sum(), "Se consideran vigentes (En curso)", (asig["Fecha Fin"].isna() & asig["Estatus_Asignacion"].eq("En curso")).sum()],
    ["Colaboradores activos sin habilidades", (activos & ~col["ID"].isin(matriz["ID Colaborador"])).sum(), "Se conservan; Tiene_Habilidades = 0", (activos & col["Tiene_Habilidades"].eq(0)).sum()],
    ["Bajas sin fecha de salida", col["Baja_Sin_Fecha"].sum(), "Se conservan vacías; Baja_Sin_Fecha = 1", col["Baja_Sin_Fecha"].sum()],
    ["Colaboradores activos con sueldo en cero", (activos & oc["Sueldo Total Integrado"].eq(0)).sum(), "Costo por hora queda vacío", (activos & col["Costo_Por_Hora"].isna()).sum()],
    ["Asignaciones con ID no encontrado", (asig["ID_Valido"] == 0).sum(), "Se identifican para revisión", (asig["ID_Valido"] == 0).sum()],
    ["Fechas guardadas como texto", sum(not pd.api.types.is_datetime64_any_dtype(oa[c]) for c in ["Fecha Inicio", "Fecha Fin"]), "Convertidas a fecha AAAA-MM-DD", sum(str(asig[c].dtype).startswith("datetime") for c in ["Fecha Inicio", "Fecha Fin"])],
], columns=["Inconsistencia", "Antes", "Tratamiento", "Registros tratados"])
mostrar(resumen, "Antes y después del tratamiento")

mostrar(pd.DataFrame({
    "Indicador": ["Colaboradores activos", "Con asignación vigente", "Disponibles",
                  "Asignaciones en curso", "Promedio costo por hora (activos)"],
    "Valor": [int(activos.sum()), int((activos & col["Proyectos_Simultaneos"].gt(0)).sum()),
              int((col["Disponible"] == "Sí").sum()), int((asig["Estatus_Asignacion"] == "En curso").sum()),
              f'${col.loc[activos, "Costo_Por_Hora"].mean():,.2f}']
}), "Atributos calculados")

mostrar(pd.DataFrame({"Atributo": tabla.columns, "Tipo de dato": tabla.dtypes.astype(str).values,
                      "Faltantes": tabla.isna().sum().values}), "Metadata de la tabla acondicionada")
mostrar(tabla.head(5), "Vista previa de la tabla acondicionada")

Inconsistencia,Antes,Tratamiento,Registros tratados
Colaboradores sin departamento,95,Se conservan; Departamento_Registrado = 0,95
Puestos PENDIENTE,48,Se conservan; Puesto_Definido = 0,48
Asignaciones sin fecha de fin,24,Se consideran vigentes (En curso),24
Colaboradores activos sin habilidades,28,Se conservan; Tiene_Habilidades = 0,28
Bajas sin fecha de salida,18,Se conservan vacías; Baja_Sin_Fecha = 1,18
Colaboradores activos con sueldo en cero,5,Costo por hora queda vacío,5
Asignaciones con ID no encontrado,0,Se identifican para revisión,0
Fechas guardadas como texto,2,Convertidas a fecha AAAA-MM-DD,2


Indicador,Valor
Colaboradores activos,75
Con asignación vigente,30
Disponibles,45
Asignaciones en curso,31
Promedio costo por hora (activos),$318.09


Atributo,Tipo de dato,Faltantes
ID Colaborador,object,0
Nómina/Rol Interno,object,0
Estatus Colaborador,object,0
Cliente,object,0
Proyecto,object,0
Rol Asignado,object,0
Tipo de Asignación,object,0
Estatus Asignación,object,0
Fecha Inicio,datetime64[ns],0
Fecha Fin,datetime64[ns],24


ID Colaborador,Nómina/Rol Interno,Estatus Colaborador,Cliente,Proyecto,Rol Asignado,Tipo de Asignación,Estatus Asignación,Fecha Inicio,Fecha Fin,Estatus_Asignacion,Dias_Restantes,ID_Valido,Estatus,Departamento,Puesto,Puesto_Definido,Costo_Por_Hora,Costo_Hora_Con_Carga,Proyectos_Simultaneos,Tiene_Habilidades
OTC004,CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR,Activo,METALSA,Asignación indefinida,PENDIENTE,Asignación,En Curso,1990-01-01 00:00:00,—,En curso,—,1,Activo,Operaciones y Soporte,CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR,1,337.48,404.98,1,0
OTC005,CONSULTOR TÉCNICO JAVA SENIOR,Activo,METALSA,Asignación indefinida,PENDIENTE,Proyecto,En Curso,1990-01-01 00:00:00,—,En curso,—,1,Activo,Operaciones y Soporte,CONSULTOR TÉCNICO JAVA SENIOR,1,381.07,457.28,1,1
OTC006,PENDIENTE,Activo,METALSA,Soporte AMS Manufacta,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,Asignación,En Curso,2025-11-03 00:00:00,—,En curso,—,1,Activo,—,PENDIENTE,0,372.23,446.68,1,1
OTC009,PENDIENTE,Activo,METALSA,Asignación indefinida,PENDIENTE,Asignación,En Curso,1990-01-01 00:00:00,—,En curso,—,1,Activo,—,PENDIENTE,0,622.42,746.90,1,1
OTC014,PENDIENTE,Activo,Grupo Daltile,Marazzi,PENDIENTE,Proyecto,En Curso,2026-01-01 00:00:00,2027-01-01 00:00:00,En curso,41.00,1,Activo,—,PENDIENTE,0,485.86,583.03,1,1


## 9. Exportar la tabla acondicionada

In [12]:
# 9. Exportar (sin datos personales)
with pd.ExcelWriter("OT_Consulting_Acondicionado.xlsx") as w:
    tabla.to_excel(w, sheet_name="Tabla_Acondicionada", index=False)
    habilidades.to_excel(w, sheet_name="Habilidades", index=False)
    resumen.to_excel(w, sheet_name="Antes_y_Despues", index=False)
print("Archivo generado: OT_Consulting_Acondicionado.xlsx")

Archivo generado: OT_Consulting_Acondicionado.xlsx
